In [11]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import ContextPrecision, ContextRecall
from src.ingestion import load_pdf
from src.embeddings import get_embedding_model, store_embeddings
from src.chunkers import header_chunker,fixed_chunker,parent_child_chunker
from src.retriever import decompose_and_retrieve
from src.reranker import rerank 
from ragas.llms import LangchainLLMWrapper 
from ragas.embeddings import LangchainEmbeddingsWrapper 
from langchain_huggingface import HuggingFaceEmbeddings 
from main import build_pipeline,run_query


/var/folders/g3/72hmy05s58748y1f4bj520xr0000gn/T/ipykernel_39481/294562493.py:7: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall
/var/folders/g3/72hmy05s58748y1f4bj520xr0000gn/T/ipykernel_39481/294562493.py:7: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall


In [2]:
load_dotenv()

# ── Setup ──────────────────────────────────────────
PDF_PATH = "langchain_rag_technical_docs_clean.pdf"

pages = load_pdf(PDF_PATH)
embedding = get_embedding_model()
header_chunks = header_chunker.chunk(pages)
header_embedding = store_embeddings(header_chunks, "Header_Chunks", embedding)
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0)

# ── Test Dataset ───────────────────────────────────
test_data = [
    {
        "question": "How would we perform RAG Evaluation and what methods are available?",
        "ground_truth": "RAG evaluation uses RAGAS with 4 metrics: Context Precision, Context Recall, Faithfulness and Answer Relevancy."
    },
    {
        "question": "What is HyDE and how does it work?",
        "ground_truth": "HyDE generates a hypothetical answer using LLM, embeds that answer and searches vector database instead of embedding raw query."
    },
    {
        "question": "What are the four RAG failure modes?",
        "ground_truth": "Four RAG failure modes are Retrieval Failure, Context Overflow, Hallucination, and Chunk Boundary Problem."
    },
    {
        "question": "What happens when chunk boundary splits an answer?",
        "ground_truth": "This is the chunk boundary problem — fix is chunk overlap or parent-child chunking"
},
{
    "question": "When should I use semantic chunking over header-based?",
    "ground_truth": "Use semantic chunking for narrative documents where topics shift mid-section like blog posts or reports"
},
]

# ── Run Pipeline & Collect Contexts ───────────────
questions = [d["question"] for d in test_data]
ground_truths = [d["ground_truth"] for d in test_data]

In [21]:
questions

['How would we perform RAG Evaluation and what methods are available?',
 'What is HyDE and how does it work?',
 'Explain Hybrid Search combining semantic similarity and keyword matching.',
 'What is Parent-Child Chunking and why is it used?',
 'What are the four RAG failure modes?']

In [3]:
contexts = []
for q in questions:
    print(f"\nRunning: {q[:50]}...")
    unique_context = decompose_and_retrieve(
        q, embedding, header_chunks, llm, header_embedding
    )
    
    # # Print topic by topic
    # print(f"Retrieved {len(unique_context)} chunks:")
    # for doc in unique_context:
    #     section = doc.metadata.get('Header 2') or doc.metadata.get('Header 1')
    #     print(f"  → {section}")
    
    reranked = rerank(q, unique_context, top_n=5)
    
    print(f"After reranking:")
    for doc in reranked:
        section = doc.metadata.get('Header 2') or doc.metadata.get('Header 1')
        score = doc.metadata.get('relevance_score')
        print(f"  → {section} | {score}")
    
    threshold = 0.05
       
    reranked = [doc for doc in reranked if doc.metadata.get('relevance_score',0) > threshold]
    
    contexts.append([doc.page_content for doc in reranked])
    print("*"*29)


    # reranked = rerank(q, unique_context, top_n=5)
    # contexts.append([doc.page_content for doc in reranked])

    # print("*"*29)


Running: How would we perform RAG Evaluation and what metho...
Rewritten query: What methods are available for RAG Evaluation and what are the core RAG metrics used in RAGAS.
Sub-questions: ['What methods are available for RAG Evaluation and what are the core RAG metrics used in RAGAS.']
[Document(metadata={'Header 1': 'Chapter 10: RAG Evaluation with RAGAS', 'source': 'uploaded_doc.pdf', 'strategy': 'header', 'header_id': 'uploaded_doc.pdf-Chapter 10: RAG Evaluation with RAGAS-10.1 The Four Core RAG Metrics-30', 'Header 2': '10.1 The Four Core RAG Metrics', 'total_chunks': 59, 'Chunk_id': 'header_30', 'hash': '0b8ee9fc8b1363e2c9232081024a7dd2', 'relevance_score': 0.17398883}, page_content='10.1 The Four Core RAG Metrics-RAGAS (RAG Assessment) measures four key dimensions of your pipeline:\n1. Context Precision: Of the chunks retrieved, how many were actually relevant?\n2. Context Recall: Of all relevant information, how much did you retrieve?\n3. Answer Faithfulness: Is the answer gr

In [35]:
questions

['How would we perform RAG Evaluation and what methods are available?',
 'What is HyDE and how does it work?',
 'Explain Hybrid Search combining semantic similarity and keyword matching.',
 'What is Parent-Child Chunking and why is it used?',
 'What are the four RAG failure modes?']

In [4]:
ragas_llm = LangchainLLMWrapper(
    ChatGroq(model='llama-3.3-70b-versatile', temperature=0.0)
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
)
# ── RAGAS Evaluation ───────────────────────────────
dataset = Dataset.from_dict({
    "question": questions,
    "contexts": contexts,
    "ground_truth": ground_truths
})
from ragas import evaluate
from ragas.metrics import ContextPrecision, ContextRecall
from datasets import Dataset

result = evaluate(
    dataset,
    metrics=[ContextPrecision(), ContextRecall()],  # ← initialize with ()
    llm =ragas_llm,
    embeddings=ragas_embeddings
)

print(result)

/var/folders/g3/72hmy05s58748y1f4bj520xr0000gn/T/ipykernel_39481/3032160031.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(
/var/folders/g3/72hmy05s58748y1f4bj520xr0000gn/T/ipykernel_39481/3032160031.py:5: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(
/var/folders/g3/72hmy05s58748y1f4bj520xr0000gn/T/ipykernel_39481/3032160031.py:15: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in 

{'context_precision': 0.8000, 'context_recall': 0.8000}


In [31]:
test_data = test_data[:3]

In [32]:
pdf_path = "langchain_rag_technical_docs_clean.pdf"
pages = load_pdf(pdf_path)

print("Loading embedding model...")
embedding = get_embedding_model()

print("Chunking...")
fixed_chunks  = fixed_chunker.chunk(pages)
header_chunks = header_chunker.chunk(pages)
child_chunks  = parent_child_chunker.create_child_chunks(header_chunks)
parent_chunks = parent_child_chunker.create_parent_chunks(child_chunks)

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 4d503a47-55f0-4bf4-8fcc-23f4faa284a9)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Loading embedding model...
Chunking...


In [ ]:
# evaluate.py — no need to build pipeline again
embedding = get_embedding_model()

# Load existing ChromaDB collections
fixed_embedding  = Chroma(collection_name="Fixed_Chunks",  embedding_function=embedding, persist_directory="./chroma_chunks_db")
header_embedding = Chroma(collection_name="Header_Chunks", embedding_function=embedding, persist_directory="./chroma_chunks_db")
child_embedding  = Chroma(collection_name="child_chunks",  embedding_function=embedding, persist_directory="./chroma_chunks_db")

In [33]:
# Step 3: For each strategy evaluate
strategies = {
    "Fixed": (fixed_embedding, fixed_chunks),
    "Header": (header_embedding, header_chunks),
    "Parent-Child": (child_embedding, child_chunks)
}

In [34]:
strategies

{'Fixed': (<langchain_chroma.vectorstores.Chroma at 0x3b67eefb0>,
  [Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'langchain_rag_technical_docs_clean.pdf', 'total_pages': 43, 'page': 3, 'page_label': '3', 'page_chapter': 'Chapter 1: Introduction to RAG', 'strategy': 'Fixed Size'}, page_content='Chapter 1: Introduction to RAG\n1.1 What is Retrieval-Augmented Generation?\nRetrieval-Augmented Generation (RAG) is an AI architecture that enhances Large Language Models (LLMs) by\nconnecting them to external knowledge sources. Instead of relying solely on knowledge encoded during\ntraining, RAG systems dynamically retrieve relevant documents at inference time.\nThe core insight behind RAG is simple: LLMs have a knowledge cutoff and a limited context window. RAG'),
   Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'langchain_rag_technical_docs_clean.pdf', 'total_pages': 43, 'page': 3, 'page_label': '3', 'pa

In [ ]:
import time
strategy_contexts = {"Fixed":[],"Header":[],"Parent-Child":[]}
ground_truths = []
questions = []

for item in test_data:
    results = run_query(
        item['question'],
        embedding,
        fixed_embedding,fixed_chunks,
        header_embedding,header_chunks,
        child_embedding,child_chunks,
        llm
        )
    
    time.sleep(15)
    
    
    questions.append(item['question'])
    ground_truths.append(item["ground_truth"])

    for strategy_name,docs in results.items():
        strategy_contexts[strategy_name].append(
            [doc.page_content for doc in docs]
        )

# Evaluate each strategy
for strategy_name, contexts in strategy_contexts.items():
    dataset = Dataset.from_dict({
        "question": questions,
        "contexts": contexts,
        "ground_truth": ground_truths
    })
    result = evaluate(
    dataset,
    metrics=[ContextPrecision(), ContextRecall()],  # ← initialize with ()
    llm =ragas_llm,
    embeddings=ragas_embeddings
)
    print(f"\n{strategy_name}: {result}")






Query: How would we perform RAG Evaluation and what methods are available?

--- Strategy: Fixed ---
Rewritten query: What methods are available for RAG Evaluation and how to perform it using RAGAS in Chapter 10.
Sub-questions: ['What methods are available for RAG Evaluation and how to perform it using RAGAS in Chapter 10.']
[Document(metadata={'hash': '4cd9c72efaf08675add27ed4bc86edc6', 'page_label': '21', 'creationdate': '', 'page_chapter': 'Chapter 10: RAG Evaluation with RAGAS', 'strategy': 'Fixed Size', 'source': 'uploaded_doc.pdf', 'page': 21, 'creator': 'PyPDF', 'producer': 'pypdf', 'total_pages': 43, 'relevance_score': 0.8705973}, page_content='Chapter 10: RAG Evaluation with RAGAS\n10.1 The Four Core RAG Metrics\nRAGAS (RAG Assessment) measures four key dimensions of your pipeline:\n1. Context Precision: Of the chunks retrieved, how many were actually relevant?\n2. Context Recall: Of all relevant information, how much did you retrieve?\n3. Answer Faithfulness: Is the answer gr

NameError: name 'context_precision' is not defined

Evaluating: 100%|██████████| 6/6 [00:44<00:00,  7.42s/it]



Fixed: {'context_precision': 0.6111, 'context_recall': 0.6667}


Evaluating: 100%|██████████| 6/6 [01:14<00:00, 12.39s/it]



Header: {'context_precision': 1.0000, 'context_recall': 1.0000}


Evaluating: 100%|██████████| 6/6 [01:37<00:00, 16.21s/it]



Parent-Child: {'context_precision': 0.8333, 'context_recall': 1.0000}


In [40]:
import pandas as pd

evl_result = pd.DataFrame([{"strategy_name":strategy_name,"result":result}])
evl_result

,strategy_name,result
0,Parent-Child,"{'context_precision': 0.8333, 'context_recall'..."
